### Homework 4 Alex Khvatov

In [1]:
from embedder import Embedder
embed = Embedder(path="models/Xenova/all-MiniLM-L6-v2")



#### Q1. Embedding a query

In [2]:
query = "How does approximate nearest neighbor search work?"
v_query = embed.encode(query)

In [3]:
v_query[0]
#-0.02

np.float64(-0.020582036807885073)

#### Loading the data

In [4]:
from gitsource import GithubRepositoryDataReader


reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
len(documents)

72

In [8]:
doc=documents[0]
print(doc)
doc.keys()

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

dict_keys(['content', 'filename'])

In [28]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


In [11]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [13]:
#Text search
from minsearch import Index

def text_search(qeury, num_results=5):
    index = Index(
            text_fields=['content'],
            keyword_fields=['filename']
        )
    index.fit(chunks)
    index.search(query, num_results=5)



In [12]:
#Vector search
from minsearch import VectorSearch

def vector_search(query, num_results=5):
    query='What metric do we use to evaluate a search engine?'
    v_query = embed.encode(query)
    v1 = np.array(v_query)
    
    vindex = VectorSearch(keyword_fields=["filename"])
    vindex.fit(X, chunks)
    result = vindex.search(v1, num_results=5)
    return result
    


In [14]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [15]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [16]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

### Q1. Generating questions

* 01-agentic-rag/lessons/01-intro.md
* 01-agentic-rag/lessons/02-environment.md
* 01-agentic-rag/lessons/03-rag.md


In [1]:
from dotenv import load_dotenv
from lmstudio import get_lmstudio_client

load_dotenv()

client = get_lmstudio_client()

In [33]:
import json

docs_to_use = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]
contents = [d for d in documents if d['filename'] in docs_to_use]

for doc in contents:
    json_content = json.dumps(doc)
    messages = [
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": json_content},
    ]
    response = client.responses.create(
            model="gpt-4o-mini",
            input=messages
        )
    print(f"Input tokens: {response.usage.input_tokens}")
    print(f"Output tokens: {response.usage.output_tokens}")
    print(f"Total tokens: {response.usage.total_tokens}")



# accumulated_content = ""
# for doc in contents:
#     accumulated_content += doc['content']




Input tokens: 1077
Output tokens: 933
Total tokens: 2010
Input tokens: 1414
Output tokens: 1506
Total tokens: 2920
Input tokens: 1900
Output tokens: 1046
Total tokens: 2946


In [2]:
from dotenv import load_dotenv
from lmstudio import get_lmstudio_client

load_dotenv()

client = get_lmstudio_client()


In [16]:
from pydantic import BaseModel, Field
class Invoice(BaseModel):
    invoice_number: int = Field(gt=0, description="invoice number to be extracted")

text = "invoice number is 1234"

In [17]:
response = client.chat.completions.parse(
    model="google/gemma-4-12b-qat",
    messages=[
        {"role": "system", "content": "Extract the invoice number from the user text."},
        {"role": "user", "content": f"Extract invoice number from this text: {text}"},
    ],
    response_format=Invoice,
)

invoice = response.choices[0].message.parsed
invoice

Invoice(invoice_number=1234)

In [20]:
print(f" response.usage.completion_tokens: {response.usage.completion_tokens}")
print(f" response.usage.prompt_tokens: {response.usage.prompt_tokens}")
print(f" response.usage.total_tokens: {response.usage.total_tokens}")
print(f" response.usage.completion_tokens_details.reasoning_tokens: {response.usage.completion_tokens_details.reasoning_tokens}")



 response.usage.completion_tokens: 101
 response.usage.prompt_tokens: 40
 response.usage.total_tokens: 141
 response.usage.completion_tokens_details.reasoning_tokens: 76


In [23]:
json_content

'[{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most commo

#### Q2. Cosine similarity

In [6]:
doc = next(
    (d for d in documents if '02-vector-search/lessons/07-sqlitesearch-vector.md' in d['filename'])
)

In [7]:
v_content = embed.encode(doc['content'])


In [8]:
#cosine similarity
import numpy as np
v1 = np.array(v_query)
v2 = np.array(v_content)
cosine_similarity = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
cosine_similarity

#0.37

np.float64(0.3610702803026061)

#### Q3. Chunking and search by hand

We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

In [10]:
vectors = []
for i in chunks:
    content = i['content']
    batch_vectors = embed.encode(content)
    vectors.extend([batch_vectors])

In [11]:
np_vectors = [np.array(v) for v in vectors]

In [12]:
np_vectors
X = np.array(np_vectors)

In [13]:
scores = X.dot(v1)
scores

array([ 3.15187667e-01,  2.01479590e-01,  5.90558833e-02, -7.67662092e-02,
        1.18452457e-01, -1.41697008e-01, -2.81406239e-02, -4.65670219e-02,
       -2.06994421e-02, -6.07744072e-02,  2.13273697e-01,  8.87601283e-02,
       -1.97269148e-02,  3.11629945e-01,  5.51034649e-01,  2.04008064e-01,
        2.12515834e-01,  1.93649166e-01,  2.51961211e-01,  1.31078657e-01,
        2.59120526e-01,  7.63816029e-02,  9.59192920e-02,  9.81469839e-03,
       -3.59106955e-02,  1.01211556e-02,  1.10786957e-01, -9.90259327e-02,
       -3.71170699e-02,  7.59057318e-02, -3.35340264e-02,  8.86838363e-03,
        1.02636359e-01,  6.89614769e-02,  1.29408804e-01,  2.57709174e-01,
        3.23680535e-01,  1.06350088e-01,  5.61891403e-02,  2.34017413e-01,
        1.97954389e-01,  9.64296226e-02,  1.93709947e-01,  2.16719278e-01,
        3.48340450e-01,  5.10906387e-02,  2.05212841e-01,  1.05416126e-01,
       -3.25432324e-02,  4.94665571e-02,  2.38574833e-01, -3.44206606e-02,
        1.82165502e-01,  

In [14]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float64(0.648901732433228))

In [15]:
chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

#### Q4. Vector search with minsearch

#### Q5. Text search vs vector search

In [19]:
query = "How do I store vectors in PostgreSQL?"

In [21]:
#Vector search


#Answer: in Vector search: 'filename': '02-vector-search/lessons/08-pgvector.md'

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

#### Q6. Hybrid search

eciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:
```
RRF(d) = sum over lists of  1 / (k + rank(d))
```

"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its 1 / (k + rank) contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant k controls how much the exact rank matters. A larger k flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller k does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.



In [22]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [23]:
query = "How do I give the model access to tools?"

In [24]:
#vector search
v_query = embed.encode(query)
v1 = np.array(v_query)
v_result = vindex.search(v1, num_results=5)

#text search
t_result = index.search(query, num_results=5)




In [25]:
results = rrf([v_result, t_result])

In [26]:
results
#01-agentic-rag/lessons/13-function-calling.md

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 